# 04 — qwen7b untreated base, TQ1_0

Select a **Colab GPU** kernel in VS Code. Read `04_quantized_setup.ipynb` for the laptop receiver, tunnel, and audible watchdog. Use only the pinned frozen base/adapters. Inference sends the same complete, unmodified teacher-row prompt with the frozen chat template, greedy decoding, seed 42, repetition penalty 1 and 1,024-token maximum. Every row is an independent llama-server completion with prompt caching disabled; prompt identity and tokenization are checked. An empty or malformed model response is still a completed response. Context-unsupported rows retain an explicit status. This condition does not use CAM-LDS importance-matrix calibration.

## 1. Restore code and data from laptop

Start the extended receiver, tunnel and watchdog using notebook 04 setup. Paste the HTTPS URL and receiver token. The receiver sends frozen code, manifests, prompts and tokenizer files; it does not send API/HF secrets in the bundle.

**Recovery / safe rerun:** New runtime or tunnel: rerun 1, then 2, 3, and the interrupted stage. The same URL/token may be reused across different models.

**Cost:** No paid API requests.



In [ ]:
from pathlib import Path
from getpass import getpass
import json, io, os, sys, urllib.request, zipfile
ROOT = Path('/content/CAM_LDS_STREAMLINED')
ROOT.mkdir(parents=True, exist_ok=True)
receiver_url = input('Local receiver HTTPS tunnel URL: ').strip().rstrip('/')
receiver_token = getpass('Local receiver token: ').strip()
from urllib.parse import urlparse
if urlparse(receiver_url).scheme != 'https':
    raise ValueError('Enter the https:// URL printed by cloudflared')
request = urllib.request.Request(receiver_url+'/bundle', headers={'Authorization':'Bearer '+receiver_token})
with urllib.request.urlopen(request, timeout=120) as response:
    bundle = response.read()
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.infolist():
        target = (ROOT/member.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Invalid bundle path')
    archive.extractall(ROOT)
(ROOT/'runtime').mkdir(exist_ok=True)
connection = ROOT/'runtime/connection.json'
connection.write_text(json.dumps({'url':receiver_url,'token':receiver_token}), encoding='utf-8')
os.chmod(connection, 0o600)
del receiver_token, bundle
sys.path.insert(0, str(ROOT))
print('Bootstrap restored. No API/HF key is stored in notebook output.')

## 2. Install pinned worker requirements

**Installations and built toolchains are reused within the same Colab runtime. A new runtime reinstalls Python packages but can restore a locally saved CUDA build for the same GPU architecture and pinned commit.** The frozen base model is downloaded directly from Hugging Face and cached within that runtime. Final quantized artifacts are saved on the laptop.

**Recovery / safe rerun:** After runtime reset rerun 1 then 2; ordinary cell reruns reuse installed packages.

**Cost:** No paid API requests.



In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-colab.txt')],check=True)
print('Worker dependencies ready')

## 3. Verify receiver, watchdog, configuration and GPU

The same authenticated receiver serves all distinct model notebooks. Check GPU identity and available memory before construction.

**Recovery / safe rerun:** After connection errors rerun 1 and 3. After a new runtime rerun 1–3. Resolve duplicate live worker leases before retrying.

**Cost:** No paid API requests.



In [ ]:
from streamlined.common import read_json, verify_experiment, CONDITIONS
from streamlined.transport import Mirror
from continuation.notebook import worker
from continuation.protocol import condition_key
import subprocess
MODEL='qwen7b'
REPRESENTATION='tq1_0'
connection=read_json(ROOT/'runtime/connection.json')
mirror=Mirror(connection['url'],connection['token'])
health=mirror.call('/health')
if not health.get('watchdog_alive'):raise RuntimeError('Start the laptop audible watchdog')
print(health, verify_experiment(ROOT)['config_hash'])
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                               '--format=csv,noheader'],text=True))

## 4. Prepare the pinned GGUF artifact

Construct TQ1_0 from the frozen untreated base. The laptop receives each final GGUF artifact and its checksum before the notebook advances. **This stage can spend substantial GPU/CPU time and local disk; completed artifacts and toolchains are reused on rerun within a live runtime.** BitNet SFT export uses a compatibility gate and stops with a clear error if this pinned native path cannot convert the merged checkpoint.

**Recovery / safe rerun:** Rerun 1–3 after a runtime change, then 4. A fully uploaded final artifact is reused; failed conversions leave no completed artifact manifest.

**Cost:** GPU/CPU construction and potentially multi-GB upload; no paid Haiku calls.



In [ ]:
worker(ROOT,MODEL,REPRESENTATION,action='prepare')

## 5. Five-row inference pilot

Run five frozen IDs per stream through the final server and save them as real results. The full run skips these IDs. The GGUF tokenizer is compared to the frozen HF tokenizer before generation. Frozen HF token IDs define the experimental input. If the GGUF pre-tokenizer segments identical text differently, the backend must detokenize the frozen IDs back to the exact rendered prompt before generation; the difference is retained in a tokenizer audit. The pilot uses the same context size as the full run. Inspect the server log for actual CUDA offload and token/s before the long run.

**Recovery / safe rerun:** Rerun 1–3 after a new runtime, then 5. Saved pilot IDs are not repeated.

**Cost:** GPU inference for up to five rows per stream.



In [ ]:
worker(ROOT,MODEL,REPRESENTATION,action='infer',pilot=5)

## 6. Complete independent inference

The untreated run covers all 990 frozen prompt IDs. Each response is durably acknowledged on the laptop as a JSON record before the next query. A CSV snapshot is refreshed every ten saved rows and at normal completion. The same prompt IDs are skipped on resume; errors are retryable. GPU, latency, token and I/O metrics are saved as telemetry.

**Recovery / safe rerun:** After interruption rerun 1–3 and 6. Rerun 4 only if the GGUF artifact was never completed; section 5 is optional on resume. The per-row JSON records, including rows after the latest CSV snapshot, are authoritative.

**Cost:** GPU inference for remaining rows; no paid Haiku calls.



In [ ]:
worker(ROOT,MODEL,REPRESENTATION,action='infer')

## 7. Verify coverage and snapshot

Read current laptop-backed record counts. Check that each required stream is terminal before launching the Haiku judge.

**Recovery / safe rerun:** Rerun 1–3 then 7 after a runtime change. Missing IDs are handled by rerunning 6.

**Cost:** No paid API requests.



In [ ]:
from continuation.store import SyncStore
from continuation.protocol import selected_rows, stream
from collections import Counter
for adapter in (CONDITIONS if 'CONDITIONS' in globals() else __import__('streamlined.common',fromlist=['CONDITIONS']).CONDITIONS) if False else (None,):
    name=stream(MODEL,REPRESENTATION,adapter)
    store=SyncStore(ROOT,name,mirror)
    store.snapshot()
    expected=len(selected_rows(ROOT,REPRESENTATION,adapter))
    states=Counter(row['status'] for row in store.rows.values())
    print(name,dict(states), 'expected:',expected,'saved:',len(store.rows))